# DASC7606 A2 · MiniGPT 实验工作台

这份 Notebook 是项目脚本的交互入口：**补全模型 → 检查 → 三次训练 → 评估 → 图表与生成分析 → 打包**。
模型实现仍在 `src/model.py`；训练、采样、评估和打包继续使用项目原有脚本，不在 Notebook 中维护第二套模型。

**当前交付不执行任何单元格，没有训练结果或示例数据。** 所有安装、下载、测试、训练和打包开关默认关闭。
首次使用按顺序运行说明与配置单元格，再按需打开对应开关；不要为了看图而重新训练。

- 报告：**4–5 页**，采用项目所有者提供的更新；官方旧 PDF 中的两页规定已过时。
- 三次完整训练：baseline、一个模型侧实验、一个优化侧实验，各从随机初始化开始，各处理 10,240,000 target tokens。
- baseline 配置保持不变，固定词表与上下文，实验只改一个主要因素，模型不超过 20M 参数。
- Mac 本地可用 CPU 检查；完整训练连接支持 BF16 的 CUDA GPU。脚本不支持 MPS。

## 使用方式

**VS Code / JupyterLab：**在项目中打开本文件，选择 Python 3.12/3.13 内核。可使用本地 `.venv`；缺少 Notebook 依赖时按项目 README 安装 `requirements-notebook.txt`。

**Colab / 远程 GPU：**把本文件放入 Notebook 环境，选择 GPU 运行时。若服务器尚无项目，把下方 `CLONE_IF_MISSING` 改为 `True`；再按需开启依赖安装和数据下载。训练输出保存在该服务器的 `outputs/`，临时运行时结束前需备份该目录，包含用于恢复的 `last.pt`。


## 1. 定位项目与运行开关

可填写绝对路径 `PROJECT_DIR`；留空时自动寻找当前项目或当前工作区中的同名克隆。
Notebook 始终通过当前内核的 `sys.executable` 调用脚本，避免使用另一个 Python 环境。
只想查看已有图表时，保持所有运行开关关闭，直接运行后面的读取与可视化单元格。


In [ ]:
from pathlib import Path
import ast
import copy
import csv
import hashlib
import html
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/Space-ash/DASC7606_A2_miniGPT.git"
PROJECT_DIR = ""  # 例如 /content/DASC7606_A2_miniGPT
CLONE_IF_MISSING = False
INSTALL_DEPENDENCIES = False
DOWNLOAD_DATA = False
RUN_SETUP_CHECK = False
RUN_PUBLIC_TESTS = False
RUN_SMOKE = False
PREPARE_EXPERIMENT_CONFIGS = False
RUN_TRAIN = {"baseline": False, "model_experiment": False, "optimizer_experiment": False}
RESUME = {"baseline": False, "model_experiment": False, "optimizer_experiment": False}
POSTPROCESS_RUNS = []  # 例：["baseline", "model_experiment", "optimizer_experiment"]
SAVE_FIGURES = False
RUN_PACKAGE = False

def is_project(path):
    return (path / "src/model.py").is_file() and (path / "configs/baseline.yaml").is_file()

if PROJECT_DIR:
    ROOT = Path(PROJECT_DIR).expanduser().resolve()
else:
    parents = [Path.cwd(), *Path.cwd().parents]
    candidates = [candidate for parent in parents
                  for candidate in (parent / "DASC7606_A2_miniGPT", parent)]
    ROOT = next((p.resolve() for p in candidates if is_project(p)),
                Path.cwd() / "DASC7606_A2_miniGPT")

if not is_project(ROOT):
    if CLONE_IF_MISSING and not ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    else:
        raise FileNotFoundError("请设置 PROJECT_DIR，或在空目标位置开启 CLONE_IF_MISSING。")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
PYTHON = sys.executable
ROLES = ("baseline", "model_experiment", "optimizer_experiment")
CONFIGS = {role: ROOT / "configs" / f"{role}.yaml" for role in ROLES}
RUN_DIRS = {role: ROOT / "outputs" / role for role in ROLES}
BASELINE_SHA256 = "a4bb0f802cca1faa0111b0cb68d97375a06f1b198649946e21085b1754362acc"

def run_command(args, env=None):
    args = list(map(str, args))
    print("Running:", " ".join(args), flush=True)
    process = subprocess.Popen(args, cwd=ROOT, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    except BaseException:
        # 中断单元格时也停止子进程，避免 GPU 训练在后台继续。
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
    if returncode:
        raise subprocess.CalledProcessError(returncode, args)

def run_python(*args):
    run_command([PYTHON, "-u", *args])

def read_json(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.is_file() else None

print("Project:", ROOT)
print("Kernel:", PYTHON)
print("Run switches:", RUN_TRAIN)


## 2. 环境、设备与固定数据

安装是可选步骤：已有合适环境时跳过。GPU 环境先按硬件安装正确的 PyTorch 2.10.0 CUDA 构建，再安装项目依赖；安装后如提示内核需重启，重启并从第 1 节重新运行。
本地克隆已复制固定数据；新 GPU 服务器可以开启 `DOWNLOAD_DATA` 获取同一版本，不能更换数据或 tokenizer。
此处的 setup 检查不调用未完成的 TODO，不代表模型正确性测试通过。


In [ ]:
if INSTALL_DEPENDENCIES:
    run_python("-m", "pip", "install", "-r", "requirements-notebook.txt")
else:
    print("跳过依赖安装；如环境缺包，请按 README 安装后选择对应内核。")
if DOWNLOAD_DATA:
    run_python("scripts/download_data.py")
if RUN_SETUP_CHECK:
    run_python("scripts/check_setup.py")


In [ ]:
import torch
import yaml
import matplotlib.pyplot as plt
from IPython.display import HTML, Image, Markdown, display

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__, "| Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0), "| BF16:", torch.cuda.is_bf16_supported())
else:
    print("当前为 CPU；可以查看已有结果，完整训练需连接 CUDA GPU。")
data_ready = all((ROOT / "data" / f).is_file()
                 for f in ("train.bin", "val.bin", "tokenizer.json", "manifest.json"))
print("Fixed data files present:", data_ready, "（哈希与依赖检查请开启 RUN_SETUP_CHECK）")

plt.rcParams.update({"figure.dpi": 115, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.2})
COLORS = dict(zip(ROLES, ("#2563eb", "#db6b22", "#13856b")))

def show_table(rows):
    if not rows:
        print("暂无结果。")
        return
    headers = list(rows[0])
    def safe(value):
        return html.escape(str(value))
    head = "".join(f"<th>{safe(k)}</th>" for k in headers)
    body = "".join("<tr>" + "".join(f"<td>{safe(row.get(k, '—'))}</td>" for k in headers)
                   + "</tr>" for row in rows)
    display(HTML("<style>.a2-table{border-collapse:collapse;font-size:13px}.a2-table th,.a2-table td"
                 "{padding:8px 12px;border:1px solid #bbb;text-align:left}</style>"
                 f"<table class='a2-table'><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

def save_figure(fig, name):
    if SAVE_FIGURES:
        dest = ROOT / "outputs/notebook_figures" / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(dest, dpi=200, bbox_inches="tight")
        print("Saved:", dest)


## 3. 实现状态、公开测试与 smoke run

先在 `src/model.py` 补全四个方法。重新导入旧内核中的模型容易造成版本混淆；这里调用新 Python 进程，使用磁盘上最新源码。
公开测试不是全部评分测试。Smoke run 只有 20 updates，输出至 `outputs/smoke`，不能代替 baseline；已存在的非空目录会由官方脚本拒绝覆盖。


In [ ]:
def unfinished_methods():
    tree = ast.parse((ROOT / "src/model.py").read_text())
    pending = []
    for cls in (n for n in tree.body if isinstance(n, ast.ClassDef)):
        for method in (n for n in cls.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))):
            if any(isinstance(n, ast.Name) and n.id == "NotImplementedError"
                   for n in ast.walk(method)):
                pending.append(f"{cls.name}.{method.name}")
    return pending

def require_implementation():
    pending = unfinished_methods()
    if pending:
        raise RuntimeError("请先完成这些 TODO：" + ", ".join(pending))

print("Unfinished methods:", unfinished_methods() or "未发现 NotImplementedError；仍需测试验证")
if RUN_PUBLIC_TESTS:
    run_python("-m", "pytest", "-q")
if RUN_SMOKE:
    require_implementation()
    run_command(["bash", "scripts/run_smoke.sh", DEVICE],
                env={**os.environ, "A2_PYTHON": PYTHON, "PYTHONUNBUFFERED": "1"})
if not RUN_PUBLIC_TESTS and not RUN_SMOKE:
    print("测试与 smoke run 均关闭。")


## 4. 运行前假设与实验配置

baseline 不修改。请先在下面填写两个实验的假设，并各选择**一个**配置字段；默认留空，不替你预先确定实验。
模型侧可以选择层数、宽度、头数或 dropout；优化侧可以选择学习率等。两个实验都分别相对 baseline 改动。

例如模型侧可填写 `{"n_layer": 4}`；优化侧可填写 `{"learning_rate": 0.0003}`。这些是可选示例，不是已经运行的实验。
开启 `PREPARE_EXPERIMENT_CONFIGS` 后生成两个 YAML 和 `outputs/experiment_plan.json`，记录运行前假设。已存在且内容不同的配置会被拒绝覆盖。


In [ ]:
MODEL_CHANGE = {}  # 例：{"n_layer": 4}
OPTIMIZER_CHANGE = {}  # 例：{"learning_rate": 0.0003}
HYPOTHESES = {
    "model_experiment": "",  # 预期结果、原因、如何判断
    "optimizer_experiment": "",
}

def token_budget(cfg):
    t = cfg["training"]
    return t["batch_size"] * t["gradient_accumulation"] * t["max_steps"] * cfg["model"]["block_size"]

def validate_run_config(role, cfg):
    from src.model import GPTConfig
    baseline = yaml.safe_load(CONFIGS["baseline"].read_text())
    if hashlib.sha256(CONFIGS["baseline"].read_bytes()).hexdigest() != BASELINE_SHA256:
        raise ValueError("原始 baseline.yaml 已改变；请先恢复官方版本。")
    GPTConfig(**cfg["model"])
    m = cfg["model"]
    c = m["n_embd"]
    parameters = m["vocab_size"] * c + m["block_size"] * c + m["n_layer"] * (12*c*c + 13*c) + 2*c
    if parameters > 20_000_000 or token_budget(cfg) != 10_240_000:
        raise ValueError("需要 <=20M 参数，且每组处理 10,240,000 target tokens。")
    if any(m[k] != baseline["model"][k] for k in ("vocab_size", "block_size")):
        raise ValueError("词表与上下文长度必须固定。")
    diffs = [(section, key) for section in ("model", "training") for key in baseline[section]
             if cfg[section].get(key) != baseline[section][key]]
    if role == "baseline" and diffs:
        raise ValueError("baseline 必须保持原始配置。")
    allowed = ({"n_layer", "n_head", "n_embd", "dropout"} if role == "model_experiment"
               else {"learning_rate", "min_lr", "warmup_steps", "weight_decay", "grad_clip"})
    section = "model" if role == "model_experiment" else "training"
    if role != "baseline" and (len(diffs) != 1 or diffs[0][0] != section or diffs[0][1] not in allowed):
        raise ValueError(f"{role} 需要仅改变一个允许的字段：{sorted(allowed)}")
    return parameters

if PREPARE_EXPERIMENT_CONFIGS:
    baseline = yaml.safe_load(CONFIGS["baseline"].read_text())
    pending = {}
    for role, section, change in (("model_experiment", "model", MODEL_CHANGE),
                                  ("optimizer_experiment", "training", OPTIMIZER_CHANGE)):
        if len(change) != 1 or not HYPOTHESES[role].strip():
            raise ValueError(f"请先填写 {role} 的单一改动与运行前假设。")
        cfg = copy.deepcopy(baseline)
        cfg[section].update(change)
        validate_run_config(role, cfg)
        if CONFIGS[role].exists() and yaml.safe_load(CONFIGS[role].read_text()) != cfg:
            raise FileExistsError(f"已有不同配置：{CONFIGS[role]}，请保留旧实验并另行规划。")
        pending[role] = cfg
    plan_path = ROOT / "outputs/experiment_plan.json"
    plan = {"hypotheses": HYPOTHESES, "configs": pending, "baseline_sha256": BASELINE_SHA256}
    if plan_path.exists() and read_json(plan_path) != plan:
        raise FileExistsError("已有不同的实验计划；请保留已有记录。")
    for role, cfg in pending.items():
        CONFIGS[role].write_text(yaml.safe_dump(cfg, sort_keys=False))
    plan_path.parent.mkdir(parents=True, exist_ok=True)
    plan_path.write_text(json.dumps(plan, ensure_ascii=False, indent=2) + "\n")
    print("实验配置与运行前计划已保存。")
else:
    print("实验配置准备关闭；先填写假设与单一改动。")

preview = []
for role, path in CONFIGS.items():
    if path.is_file():
        cfg = yaml.safe_load(path.read_text())
        preview.append({"run": role, "layers": cfg["model"]["n_layer"],
                        "width": cfg["model"]["n_embd"], "peak LR": cfg["training"]["learning_rate"],
                        "target tokens": token_budget(cfg), "config": path.name})
show_table(preview)


## 5. 连接 GPU 后正式训练

将第 1 节中对应的 `RUN_TRAIN` 开关设为 `True`，再运行本节。三组可分别启动，也可在 GPU 时间足够时依次运行。
这会调用官方 `python -m src.train`，实时输出日志。默认要求 CUDA 和 BF16，避免误在 CPU 上启动完整训练。

只有同一 run 因中断需要恢复时，才开启相应 `RESUME`；它会使用该目录自己的 `last.pt`。
新实验必须使用随机初始化，不得从另一个 run 的 checkpoint 开始。正式训练完成后可将开关关回 `False`。


In [ ]:
for role in ROLES:
    if not RUN_TRAIN[role]:
        continue
    require_implementation()
    if DEVICE != "cuda" or not torch.cuda.is_bf16_supported():
        raise RuntimeError("完整训练需要支持 BF16 的 CUDA GPU；请更换运行时后重新运行设备检测。")
    if not CONFIGS[role].is_file():
        raise FileNotFoundError(f"请先创建实验配置：{CONFIGS[role]}")
    from src.utils import load_config
    cfg = load_config(CONFIGS[role])
    validate_run_config(role, cfg)
    if role != "baseline":
        plan = read_json(ROOT / "outputs/experiment_plan.json")
        if not plan or plan["configs"].get(role) != cfg or not plan["hypotheses"].get(role, "").strip():
            raise ValueError("实验配置与运行前假设必须先保存，且与即将运行的配置一致。")
    args = ["-m", "src.train", "--config", CONFIGS[role], "--out_dir", RUN_DIRS[role], "--device", DEVICE]
    if RESUME[role]:
        args.append("--resume")
    run_python(*args)
if not any(RUN_TRAIN.values()):
    print("全部正式训练开关关闭；没有启动训练。")


## 6. 生成官方曲线、固定提示词续写与完整验证集评估

训练后在第 1 节的 `POSTPROCESS_RUNS` 中选择要处理的 run。每组调用三个现有脚本，生成提交需要的证据文件。
采样统一为 temperature 0.8、top-k 40、seed 2026、最多 160 个新 token，使用官方固定提示词。
评估显式使用 `--max-tokens 0`，得到完整公开验证集的 FP32 结果；不能与 trainer 的抽样混合精度验证混为同一协议。
只有这节会重新生成样本与评估文件；后面的展示单元格只读取已有结果。


In [ ]:
for role in POSTPROCESS_RUNS:
    if role not in ROLES:
        raise ValueError(f"未知 run：{role}")
    require_implementation()
    directory = RUN_DIRS[role]
    checkpoint = directory / "best.pt"
    if not checkpoint.is_file():
        raise FileNotFoundError(f"请先完成该组训练：{checkpoint}")
    run_python("scripts/plot_run.py", directory)
    run_python("-m", "src.sample", "--checkpoint", checkpoint, "--out", directory / "samples.json",
               "--device", DEVICE, "--temperature", "0.8", "--top-k", "40", "--seed", "2026",
               "--max-new-tokens", "160")
    run_python("-m", "src.evaluate", "--checkpoint", checkpoint, "--tokens", "data/val.bin",
               "--max-tokens", "0", "--device", DEVICE, "--out", directory / "evaluation.json")
if not POSTPROCESS_RUNS:
    print("未选择后处理 run；直接读取已有结果即可。")


## 7. 结果总览与提交证据检查

训练与后处理完成后，重新运行本节及后续图表单元格即可更新展示。
表格明确区分 run 是否完成、trainer 最佳抽样 loss，以及独立全量 FP32 loss/perplexity。
缺少文件显示“—”，不会填入假数据。八个证据文件齐全只代表文件存在，最终仍需官方提交检查。


In [ ]:
EVIDENCE = ("metrics.json", "config.json", "environment.json", "train.csv", "eval.jsonl",
            "samples.json", "loss_curve.png", "evaluation.json")
metrics_by_role = {role: read_json(path / "metrics.json") for role, path in RUN_DIRS.items()}
eval_by_role = {role: read_json(path / "evaluation.json") for role, path in RUN_DIRS.items()}
rows = []
for role, directory in RUN_DIRS.items():
    m = metrics_by_role[role] or {}
    e = eval_by_role[role] or {}
    env = read_json(directory / "environment.json") or {}
    f = lambda x: f"{x:.4f}" if isinstance(x, (int, float)) else "—"
    rows.append({"run": role, "status": m.get("status", "not started"),
                 "updates": m.get("step", "—"), "target tokens": m.get("train_tokens", "—"),
                 "parameters": m.get("parameter_count", "—"),
                 "trainer best loss": f(m.get("best_val_loss")),
                 "FP32 eval loss": f(e.get("cross_entropy")), "FP32 PPL": f(e.get("perplexity")),
                 "eval targets": e.get("evaluated_tokens", "—"),
                 "runtime min": f(m["runtime_seconds"] / 60) if "runtime_seconds" in m else "—",
                 "device": env.get("device", "—"),
                 "evidence": f"{sum((directory / name).is_file() for name in EVIDENCE)}/8"})
show_table(rows)


## 8. 学习曲线：相同 token budget 下的收敛比较

横轴为已处理的训练目标 token 数，便于公平比较。左图为固定抽样训练 loss，右图为固定抽样验证 loss，均来自 `eval.jsonl`；不是独立全量 FP32 评估。
断点恢复可能产生相同步数记录，这里保留每个 token 位置最后一条记录。
开启 `SAVE_FIGURES` 可把图另存到 `outputs/notebook_figures/`，用于报告；不改变各 run 的官方证据。


In [ ]:
def load_eval_curve(directory):
    path = directory / "eval.jsonl"
    if not path.is_file():
        return []
    rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    by_tokens = {r["train_tokens"]: r for r in rows}
    return [by_tokens[k] for k in sorted(by_tokens)]

curves = {role: load_eval_curve(path) for role, path in RUN_DIRS.items()}
if any(curves.values()):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
    for role, points in curves.items():
        if not points:
            continue
        x = [p["train_tokens"] / 1e6 for p in points]
        for ax, key in zip(axes, ("train_loss", "val_loss")):
            ax.plot(x, [p[key] for p in points], label=role, color=COLORS[role], marker="o", ms=3)
    for ax, title in zip(axes, ("Sampled training loss", "Sampled validation loss")):
        ax.set(title=title, xlabel="Processed target tokens (millions)", ylabel="Cross-entropy (nats)")
        ax.legend(fontsize=8)
    fig.tight_layout()
    save_figure(fig, "learning_curves.png")
    plt.show()
else:
    print("暂无 eval.jsonl。完成训练后重新运行本单元格即可显示学习曲线。")


## 9. 训练诊断：学习率与单步吞吐

从 `train.csv` 读取记录。学习率有助于解释 warmup 和衰减过程；吞吐为**被记录的单次 update**的 tokens/seconds，不是整个 run 的平均吞吐，也不包含完整评估开销。
不同硬件的速度不可直接归因于模型或优化改动；报告同时列出设备与 `metrics.json` 的总运行时间。


In [ ]:
logs = {}
for role, directory in RUN_DIRS.items():
    path = directory / "train.csv"
    if path.is_file():
        with path.open(newline="") as handle:
            logs[role] = list(csv.DictReader(handle))
if any(logs.values()):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for role, points in logs.items():
        if not points:
            continue
        axes[0].plot([int(p["step"]) for p in points], [float(p["lr"]) for p in points],
                     color=COLORS[role], label=role)
        cfg = read_json(RUN_DIRS[role] / "config.json")
        if cfg:
            per_step = cfg["training"]["batch_size"] * cfg["training"]["gradient_accumulation"] * cfg["model"]["block_size"]
            valid = [p for p in points if float(p["seconds"]) > 0]
            axes[1].plot([int(p["step"]) for p in valid],
                         [per_step / float(p["seconds"]) for p in valid], color=COLORS[role], label=role)
    axes[0].set(title="Learning-rate schedule", xlabel="Optimizer update", ylabel="Learning rate")
    axes[1].set(title="Throughput of logged updates", xlabel="Optimizer update", ylabel="Target tokens / second")
    for ax in axes:
        if ax.lines:
            ax.legend(fontsize=8)
    fig.tight_layout()
    save_figure(fig, "training_diagnostics.png")
    plt.show()
else:
    print("暂无 train.csv。完成训练后可查看学习率与吞吐。")


## 10. 完整 FP32 评估与成本对比

仅在结果精度、协议、context、评估 token 数与数据哈希一致，且覆盖完整验证集时绘制 loss/perplexity 比较。
柱形图纵轴从零开始。散点图结合参数量和总运行时间观察代价；固定训练 token 数不代表固定计算成本。


In [ ]:
# 每次刷新都重新读取磁盘结果。
full = {role: read_json(path / "evaluation.json") for role, path in RUN_DIRS.items()}
full = {role: e for role, e in full.items() if e}
manifest = read_json(ROOT / "data/manifest.json")
protocol_keys = ("precision", "protocol", "context_length", "evaluated_tokens", "tokens_sha256")
comparable = bool(full) and all(all(k in e for k in protocol_keys) and e["precision"] == "float32"
                              for e in full.values())
comparable = comparable and len({tuple(e[k] for k in protocol_keys) for e in full.values()}) == 1
comparable = comparable and bool(manifest) and all(
    e["evaluated_tokens"] == manifest["val_tokens"] - 1
    and e["tokens_sha256"] == manifest["files"]["val.bin"]["sha256"] for e in full.values())
if not full:
    print("暂无 evaluation.json；训练后执行第 6 节的完整评估。")
elif not comparable:
    print("评估文件未满足相同且完整的 FP32 协议，暂不绘制对比；请重新进行统一全量评估。")
else:
    labels = list(full)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for ax, key, title in zip(axes[:2], ("cross_entropy", "perplexity"),
                              ("Full FP32 validation loss", "Full FP32 perplexity")):
        bars = ax.bar(labels, [full[r][key] for r in labels], color=[COLORS[r] for r in labels])
        ax.set(title=title, ylabel=key, ylim=(0, None))
        ax.tick_params(axis="x", rotation=22)
        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
        ax.margins(y=0.15)
    for role in labels:
        m = read_json(RUN_DIRS[role] / "metrics.json") or {}
        if "runtime_seconds" in m:
            axes[2].scatter(m["runtime_seconds"] / 60, full[role]["cross_entropy"],
                            s=max(35, m.get("parameter_count", 0) / 100_000), color=COLORS[role])
            axes[2].annotate(role, (m["runtime_seconds"] / 60, full[role]["cross_entropy"]),
                             xytext=(5, 5), textcoords="offset points", fontsize=8)
    axes[2].set(title="Loss vs total runtime (size = parameters)", xlabel="Total run time (minutes)",
                ylabel="Full FP32 validation loss")
    fig.tight_layout()
    save_figure(fig, "evaluation_comparison.png")
    plt.show()
    if "baseline" in full:
        base = full["baseline"]["cross_entropy"]
        show_table([{"run": role, "loss minus baseline": f"{full[role]['cross_entropy'] - base:+.4f}",
                     "interpretation": "lower is better; descriptive single-seed comparison"} for role in labels])


## 11. 固定提示词的故事续写

从 `samples.json` 读取已有文本，不在展示时重新采样。各组使用相同 prompt 和采样参数，比较人物、事件因果、重复、结尾及提示词遵循情况。
语法通顺不能单独证明故事连贯。以下按 prompt 对齐显示，并标明 checkpoint 和采样设置是否一致；没有文件时只提示等待结果。


In [ ]:
sample_files = {role: read_json(path / "samples.json") for role, path in RUN_DIRS.items()}
sample_files = {role: data for role, data in sample_files.items() if data}
if not sample_files:
    print("暂无 samples.json。完成第 6 节采样后，本节会按相同提示词展示三组续写。")
else:
    keys = ("seed", "temperature", "top_k", "max_new_tokens")
    show_table([{"run": role, **{key: data.get(key, "—") for key in keys},
                 "matches evaluation checkpoint": (
                     data.get("checkpoint_sha256") == (read_json(RUN_DIRS[role] / "evaluation.json") or {}).get("checkpoint_sha256")
                     if data.get("checkpoint_sha256") and (RUN_DIRS[role] / "evaluation.json").is_file() else "not checked")}
                for role, data in sample_files.items()])
    if len({tuple(data.get(k) for k in keys) for data in sample_files.values()}) > 1:
        print("注意：采样设置不同，请统一参数后再比较。")
    prompts = list(dict.fromkeys(sample["prompt"] for data in sample_files.values() for sample in data["samples"]))
    for index, prompt in enumerate(prompts, 1):
        display(HTML(f"<h3>Prompt {index}</h3><blockquote>{html.escape(prompt)}</blockquote>"))
        for role, data in sample_files.items():
            sample = next((s for s in data["samples"] if s["prompt"] == prompt), None)
            if sample:
                text = html.escape(sample["continuation"])
                display(HTML(f"<b>{html.escape(role)}</b><pre style='white-space:pre-wrap;line-height:1.6;padding:12px;border:1px solid #aaa'>{text}</pre>"))
            else:
                print(role, "缺少此 prompt 的结果。")


## 12. 观察记录与 4–5 页报告

在实验后填写以下内容，不要先写结论：

| 观察维度 | Baseline | 模型侧实验 | 优化侧实验 |
|---|---|---|---|
| 收敛速度与验证曲线 | 待观察 | 待观察 | 待观察 |
| 完整 FP32 loss / perplexity | 待观察 | 待观察 | 待观察 |
| 故事人物与情节一致性 | 待观察 | 待观察 | 待观察 |
| 重复、矛盾、提前结束 | 待观察 | 待观察 | 待观察 |
| 假设是否得到支持 | 参照 | 待分析 | 待分析 |

建议报告结构：方法 1 页、实验设计 1 页、定量结果 1 页、生成分析 1 页、讨论与复现最多 1 页；写四页时合并最后两项。
参见 `docs/REPORT_OUTLINE.md` 与 `docs/EXPERIMENTS.md`。报告放代表性片段，完整文本保留在各 run 的 `samples.json`。
记录真实命令、环境、seed、Git commit、评估协议和局限；只有一个 seed 时，不声称统计显著性。


## 13. 选择最终模型并调用官方打包脚本

仅根据公开验证集选择最终 checkpoint。先完成报告并导出到项目根目录 `report.pdf`；填写真实学号与所选 run，再开启 `RUN_PACKAGE`。
三个 run 的证据都要提交，但只提交一个 `best.pt`。最终 ZIP 不超过 100 MB；代码中的学号和运行产物不必公开到 GitHub。

Notebook 是辅助工作台，官方打包脚本不自动包含它；正式复现需要的模型代码保留在 `src/`，命令写进 README。Moodle 只有一次最终提交机会，保存草稿不算提交。


In [ ]:
STUDENT_ID = ""  # 仅在本地填写真实学号，上传公开 Notebook 前清空
SELECTED_RUN = "baseline"  # 根据公开验证结果选择
if RUN_PACKAGE:
    if not STUDENT_ID.isdigit() or SELECTED_RUN not in ROLES:
        raise ValueError("请填写真实学号与有效的 SELECTED_RUN。")
    require_implementation()
    if not (ROOT / "report.pdf").is_file():
        raise FileNotFoundError("请先导出 4–5 页报告到项目根目录 report.pdf。")
    for role, directory in RUN_DIRS.items():
        m = read_json(directory / "metrics.json") or {}
        if m.get("status") != "complete":
            raise ValueError(f"{role} 尚未完成。")
        missing = [name for name in EVIDENCE if not (directory / name).is_file()]
        if missing:
            raise FileNotFoundError(f"{role} 缺少证据：{missing}")
    archive = ROOT / "outputs/submissions" / f"{STUDENT_ID}.zip"
    run_python("scripts/package_submission.py", "--student-id", STUDENT_ID,
               "--baseline", RUN_DIRS["baseline"], "--model-experiment", RUN_DIRS["model_experiment"],
               "--optimizer-experiment", RUN_DIRS["optimizer_experiment"], "--select", SELECTED_RUN,
               "--report", "report.pdf", "--out", archive)
    run_python("scripts/validate_submission.py", archive)
    size = archive.stat().st_size
    if size > 100_000_000:
        raise ValueError(f"ZIP 为 {size / 1e6:.2f} MB，超过 100 MB；请调整报告图片等再重新打包。")
    print(f"提交文件：{archive} ({size / 1e6:.2f} MB)")
else:
    print("打包关闭；当前没有创建提交 ZIP。")


## 保存与继续工作

- Notebook 文件应保留说明和代码；公开到 GitHub 前清空输出与真实学号，避免把个人信息或大体积输出提交到仓库。
- 断开 GPU 前备份 `outputs/`（包含恢复训练用的 `last.pt`）及新实验配置；数据可以重新下载，训练输出不能凭空恢复。
- 每次修改模型后先测试，再开始新 run；查看旧结果无需运行训练单元格。
- 当前 Notebook 的代码未执行，执行计数与输出均为空。训练结果将在你后续完成 TODO 并连接 GPU 运行后显示。
